# 0. Imports y Configuración

In [3]:
import math
from pathlib import Path
import cv2
from tqdm.auto import tqdm

In [4]:
# Ruta del video que se desea procesar
RUTA_VIDEO = Path("Videos") / "video30min-11to22.mp4"

# Cantidad de imágenes que se extraerán por cada segundo de video
FRAMES_POR_SEGUNDO = 5.0

# Intervalo opcional del video
# None significa comenzar desde el inicio o terminar al final
SEGUNDO_INICIO = None
SEGUNDO_FIN = None

# Compresión PNG entre 0 y 9
COMPRESION_PNG = 3

# Carpeta donde se guardarán los frames
CARPETA_FRAMES = RUTA_VIDEO.parent / RUTA_VIDEO.stem / "Frames"

# 1. Extracción Frames

In [ ]:
captura = cv2.VideoCapture(str(RUTA_VIDEO))  # Abre el video

fps_video = captura.get(cv2.CAP_PROP_FPS)  # Obtiene los FPS originales
total_frames = captura.get(cv2.CAP_PROP_FRAME_COUNT)  # Obtiene el total de frames
duracion = total_frames / fps_video  # Calcula la duración en segundos

inicio = 0.0 if SEGUNDO_INICIO is None else SEGUNDO_INICIO  # Define el segundo inicial
fin = duracion if SEGUNDO_FIN is None else SEGUNDO_FIN  # Define el segundo final

cantidad_salida = math.ceil((fin - inicio) * FRAMES_POR_SEGUNDO)  # Calcula la cantidad de frames de salida
intervalo = 1 / FRAMES_POR_SEGUNDO  # Calcula los segundos entre cada frame
cantidad_digitos = len(str(cantidad_salida - 1))  # Calcula los dígitos según el último índice

CARPETA_FRAMES.mkdir(parents=True, exist_ok=True)  # Crea la carpeta de salida

for indice in tqdm(range(cantidad_salida), desc="Extrayendo frames", unit="frame"):
    segundo = inicio + indice * intervalo  # Calcula el momento que se extraerá

    captura.set(cv2.CAP_PROP_POS_MSEC, segundo * 1000)  # Mueve el video al momento calculado
    lectura_correcta, frame = captura.read()  # Lee el frame correspondiente

    if not lectura_correcta:
        captura.release()  # Cierra el video antes de detener la ejecución
        raise RuntimeError(f"No se pudo leer el segundo {segundo:.2f}")

    nombre = f"{indice:0{cantidad_digitos}d}.png"  # Crea el nombre consecutivo
    ruta_salida = CARPETA_FRAMES / nombre  # Construye la ruta completa

    cv2.imwrite(str(ruta_salida), frame, [cv2.IMWRITE_PNG_COMPRESSION, COMPRESION_PNG])  # Guarda el frame

captura.release()  # Cierra el video y libera los recursos

print(f"Extracción terminada: {cantidad_salida} frames guardados en {CARPETA_FRAMES}")

Extrayendo frames:  55%|█████▍    | 1800/3300 [1:45:41<17:21:37, 41.66s/frame]   